In [17]:
from pathlib import Path

import numpy as np
import pandas as pd
from power_grid_model import ComponentType, PowerGridModel
from power_grid_model.utils import json_deserialize
from power_grid_model.validation import errors_to_string, validate_batch_data, validate_input_data


class InputDataValidationError(ValueError):
    pass


class BatchDataValidationError(ValueError):
    pass


class ProfileTimestampMismatchError(ValueError):
    pass


class ProfileLoadIDMismatchError(ValueError):
    pass


class TimeIndexLengthError(ValueError):
    pass


def read_json_file(file_path):
    return Path(file_path).read_text()


def deserialize_pgm_json(json_data):
    return json_deserialize(json_data)


def create_pgm(input_data):
    errors = validate_input_data(input_data)
    if errors:
        raise InputDataValidationError(errors_to_string(errors, name="input data", details=True))
    return PowerGridModel(input_data)


def read_load_profile(file_path):
    return pd.read_parquet(file_path)


def validate_load_profile(active_profile, reactive_profile):
    if active_profile.empty or reactive_profile.empty:
        raise ValueError("Load profiles cannot be empty.")
    if not active_profile.index.equals(reactive_profile.index):
        raise ProfileTimestampMismatchError("Active and reactive profiles have different timestamps.")
    if not active_profile.index.is_unique or not active_profile.index.is_monotonic_increasing:
        raise ProfileTimestampMismatchError("Load profile timestamps must be unique and sorted.")
    if not active_profile.columns.equals(reactive_profile.columns):
        raise ProfileLoadIDMismatchError("Active and reactive profiles have different load IDs.")
    if not active_profile.columns.is_unique:
        raise ProfileLoadIDMismatchError("Load profile IDs must be unique.")


def create_load_batch_update(active_profile, reactive_profile):
    validate_load_profile(active_profile, reactive_profile)

    load_ids = np.tile(
        active_profile.columns.to_numpy(dtype=np.int32),
        (len(active_profile), 1),
    )

    return {
        ComponentType.sym_load: {
            "id": load_ids,
            "p_specified": active_profile.to_numpy(),
            "q_specified": reactive_profile.to_numpy(),
        }
    }


def run_batch_power_flow(input_data, batch_update):
    errors = validate_batch_data(input_data, batch_update)
    if errors:
        raise BatchDataValidationError(errors_to_string(errors, name="batch update", details=True))
    model = PowerGridModel(input_data)
    return model.calculate_power_flow(update_data=batch_update)


def aggregate_node_voltage_results(results, time_index):
    node_results = results[ComponentType.node]
    if len(node_results) != len(time_index):
        raise TimeIndexLengthError("The number of timestamps does not match the number of result batches.")

    rows = []
    for i, timestamp in enumerate(time_index):
        ids = node_results[i]["id"]
        u_pu = node_results[i]["u_pu"]

        max_idx = np.argmax(u_pu)
        min_idx = np.argmin(u_pu)

        rows.append(
            {
                "timestamp": timestamp,
                "max_u_pu": u_pu[max_idx],
                "max_u_pu_node_id": ids[max_idx],
                "min_u_pu": u_pu[min_idx],
                "min_u_pu_node_id": ids[min_idx],
            }
        )
    return pd.DataFrame(rows).set_index("timestamp")


def aggregate_line_results(results, time_index):
    line_results = results[ComponentType.line]
    if len(line_results) != len(time_index):
        raise TimeIndexLengthError("The number of timestamps does not match the number of result batches.")

    line_ids = line_results[0]["id"]

    rows = []
    for line_index, line_id in enumerate(line_ids):
        loading = line_results["loading"][:, line_index]
        max_idx = np.argmax(loading)
        min_idx = np.argmin(loading)
        loss_w = line_results["p_from"][:, line_index] + line_results["p_to"][:, line_index]
        hours = (time_index - time_index[0]).total_seconds() / 3600
        energy_loss_kwh = np.trapezoid(loss_w, hours) / 1000
        rows.append(
            {
                "line_id": line_id,
                "max_loading_pu": loading[max_idx],
                "max_loading_timestamp": time_index[max_idx],
                "min_loading_pu": loading[min_idx],
                "min_loading_timestamp": time_index[min_idx],
                "energy_loss_kwh": energy_loss_kwh,
            }
        )

    return pd.DataFrame(rows).set_index("line_id")


# 📘 Assignment 2 – Power Grid Simulation

## 🧩 Overview
In this assignment, we simulate an electrical power grid over time using computational models.

A power grid is a network of nodes and connections that transport electricity from generation to consumers.

### 🎯 Goals
- Validate grid input data
- Process load profiles
- Run batch simulations
- Extract insights

---

# 🌐 Power Flow Concept

## What is Power Flow?
Power flow analysis determines how electricity moves through a network.

It calculates:
- Voltages at nodes
- Power flows in lines
- Network losses

👉 It answers: *Can the grid safely operate under given conditions?*

# ⚙️ Step 1 – Input Handling

We load JSON data and convert it into a simulation model.

### Why validation matters:
- Prevents invalid networks
- Ensures safe simulation
- Detects structural issues early

In [18]:
from pathlib import Path
import numpy as np
import pandas as pd
from power_grid_model import ComponentType, PowerGridModel
from power_grid_model.utils import json_deserialize
from power_grid_model.validation import errors_to_string, validate_batch_data, validate_input_data

## ⚠️ Custom Exceptions
Clear error handling improves robustness.

In [19]:
class InputDataValidationError(ValueError): pass
class BatchDataValidationError(ValueError): pass
class ProfileTimestampMismatchError(ValueError): pass
class ProfileLoadIDMismatchError(ValueError): pass
class TimeIndexLengthError(ValueError): pass

# 📊 Step 2 – Load Profiles

Load profiles define how demand changes over time.

### Requirements:
- Matching timestamps
- Matching load IDs
- Sorted and unique data

### Why this matters:
Incorrect data → incorrect physics → unreliable results

In [20]:
def read_load_profile(path):
    return pd.read_parquet(path)

def validate_load_profile(a, r):
    if a.empty or r.empty: raise ValueError('empty')
    if not a.index.equals(r.index): raise ProfileTimestampMismatchError()
    if not a.columns.equals(r.columns): raise ProfileLoadIDMismatchError()

# 🔁 Step 3 – Batch Processing

We transform profiles into a simulation-ready format.

### Key idea:
Run many timesteps efficiently in one call.

In [21]:
def create_load_batch_update(a,r):
    validate_load_profile(a,r)
    ids=np.tile(a.columns.to_numpy(dtype=np.int32),(len(a),1))
    return {ComponentType.sym_load:{'id':ids,'p_specified':a.to_numpy(),'q_specified':r.to_numpy()}}

# ⚡ Step 4 – Simulation

The model computes voltages and flows at each timestep.

In [22]:
def run_batch_power_flow(input_data,batch):
    errors=validate_batch_data(input_data,batch)
    if errors: raise BatchDataValidationError()
    return PowerGridModel(input_data).calculate_power_flow(update_data=batch)

# 📈 Step 5 – Node Analysis

We extract:
- Max voltage
- Min voltage

### Why:
Voltage limits determine grid stability.

In [23]:
def aggregate_node_voltage_results(results,time_index):
    nodes=results[ComponentType.node]
    rows=[]
    for i,t in enumerate(time_index):
        ids=nodes[i]['id']; u=nodes[i]['u_pu']
        rows.append({'timestamp':t,'max':u.max(),'min':u.min()})
    return pd.DataFrame(rows).set_index('timestamp')

# 📉 Step 6 – Line Analysis

We compute:
- Loading extremes
- Energy loss

### Trapezoidal Rule
We approximate energy by integrating power over time.

In [24]:
def aggregate_line_results(results,time_index):
    line=results[ComponentType.line]
    hours=(time_index-time_index[0]).total_seconds()/3600
    ids=line[0]['id']
    rows=[]
    for i,lid in enumerate(ids):
        loading=line['loading'][:,i]
        loss=line['p_from'][:,i]+line['p_to'][:,i]
        rows.append({'line_id':lid,'energy_loss_kwh':np.trapezoid(loss,hours)/1000})
    return pd.DataFrame(rows).set_index('line_id')

# ✅ Example Pipeline

In [25]:
# Example usage (requires data files)
input_data=deserialize_pgm_json(read_json_file('data/input_network_data.json'))
active=read_load_profile('data/active_power_profile.parquet')
reactive=read_load_profile('data/reactive_power_profile.parquet')
batch=create_load_batch_update(active,reactive)
results=run_batch_power_flow(input_data,batch)

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

# ✅ Conclusion

### Key Takeaways
- Power grids behave as networks
- Time-series simulation reveals dynamic behavior
- Validation is essential
- Numerical integration enables energy calculations